# FitLoop - Part 1: Data cleaning

**Goal.** `users.csv` and `marketing_spend.csv` are a raw production extract (Jan 2025 - Jun 2026, extracted 30 June 2026). Before measuring CAC, retention or LTV we make sure the data can be trusted. Dirty data would give wrong unit economics and a wrong recommendation to the board.

**Method.**
- The raw files are never modified: we work on copies.
- One rule at a time; after each rule we print how many rows it affected.
- We **remove** only what is certainly wrong (duplicates, test accounts, impossible dates). Ambiguous or missing values are **fixed or flagged**, never silently dropped.

| Step | What we do |
|---|---|
| 0 | Load raw data, set up the log |
| 1 | Fix formats (dates, types) |
| 2 | Standardise labels |
| 3 | Remove duplicates |
| 4 | Remove test / internal accounts |
| 5 | Remove impossible values |
| 6 | Handle missing values |
| 7 | Clean the marketing spend file |
| 8 | Flag censored cohorts |
| 9 | Check and save |

## Step 0 - Load raw data and set up the log

**Why.** Keep an untouched copy of the raw data. A small helper prints how many rows each rule affects (we are not keeping a formal log for now).

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "."                      # folder containing the two CSV files
EXTRACTION_DATE = pd.Timestamp("2026-06-30")
START_DATE = pd.Timestamp("2025-01-01")

users_raw = pd.read_csv(f"{DATA_DIR}/users.csv", dtype=str)       # read everything as text first
spend_raw = pd.read_csv(f"{DATA_DIR}/marketing_spend.csv", dtype=str)

users = users_raw.copy()
spend = spend_raw.copy()

def parse_dates(series):
    # ISO dates (YYYY-MM-DD) are parsed first and unambiguously; everything else is read as DD/MM/YYYY
    s = series.astype("string").str.strip()
    iso = pd.to_datetime(s, format="ISO8601", errors="coerce")
    other = pd.to_datetime(s.where(iso.isna()), format="mixed", dayfirst=True, errors="coerce")
    return iso.fillna(other)

def log_rule(step, table, rule, action, before, after, affected, comment=""):
    # simple printout only (no log kept): what we did and how many rows it touched
    print(f"[{step}] {table}: {rule} -> {action}, {int(affected)} rows")

print("users :", users_raw.shape, "| spend:", spend_raw.shape)
users.head()

## Step 1 - Fix formats (dates and types)

**Why.** Dates arrive in mixed formats and as text. Without proper dates we cannot build cohorts, trial-to-paid delays or churn.

**Comment.** We parse the 4 date columns. ISO dates (YYYY-MM-DD) are read first, the remaining ones as DD/MM/YYYY. A value that was filled in but cannot be parsed is counted and flagged (not deleted). Check manually that DD/MM vs MM/DD is not ambiguous in the raw values.

In [ ]:
date_cols = ["signup_date", "trial_start_date", "subscription_start_date", "subscription_end_date"]

# quick look at the raw formats
for c in date_cols:
    print(c, "->", users[c].dropna().str.replace(r"\d", "9", regex=True).value_counts().head(5).to_dict())

In [ ]:
before = len(users)
for c in date_cols:
    users[c] = users[c].str.strip().replace("", np.nan)
    filled = users[c].notna()
    parsed = parse_dates(users[c])
    n_bad = int((filled & parsed.isna()).sum())          # filled but unparseable
    n_fixed = int((filled & parsed.notna()).sum())
    users[c] = parsed
    log_rule("1. Formats", "users", f"Parse {c} to datetime (mixed formats)", "Fix",
             before, len(users), n_fixed,
             f"{n_bad} filled values could not be parsed -> set to empty (NaT)")

# numeric / flag columns
users["activated_d7"] = pd.to_numeric(users["activated_d7"], errors="coerce")
users.dtypes

## Step 2 - Standardise labels

**Why.** `fb`, `Facebook` and `META ` are the same channel. If not unified, CAC by channel and the like-for-like comparisons are wrong.

**Comment.** Text is trimmed and lower-cased, then mapped to one canonical label. Anything not mapped is printed so we can decide by hand, never guessed.

In [ ]:
def map_labels(series, rules):
    """rules: list of (substring, canonical). Unmapped -> kept as cleaned text and reported."""
    s = series.astype("string").str.strip().str.lower()
    out = pd.Series(pd.NA, index=series.index, dtype="string")
    for key, canon in rules:
        out = out.mask(out.isna() & s.str.contains(key, na=False), canon)
    unmapped = s[out.isna() & s.notna()].value_counts()
    return out, unmapped

CHANNEL_RULES = [("face", "Meta"), ("fb", "Meta"), ("meta", "Meta"), ("insta", "Meta"),
                 ("goog", "Google"), ("adwords", "Google"),
                 ("tik", "TikTok"),
                 ("influ", "Influencer"),
                 ("organ", "Organic"), ("seo", "Organic"),
                 ("refer", "Referral"), ("friend", "Referral")]
COUNTRY_RULES = [("fr", "FR"), ("be", "BE"), ("ch", "CH"), ("swit", "CH"), ("lu", "LU")]
PLATFORM_RULES = [("ios", "iOS"), ("iphone", "iOS"), ("android", "Android")]
PLAN_RULES = [("month", "monthly"), ("annu", "annual"), ("year", "annual")]

print("BEFORE:", {c: sorted(users[c].dropna().unique().tolist())[:15] for c in ["channel", "country", "platform", "plan"]})

In [ ]:
before = len(users)
for col, rules in [("channel", CHANNEL_RULES), ("country", COUNTRY_RULES),
                   ("platform", PLATFORM_RULES), ("plan", PLAN_RULES)]:
    old = users[col].copy()
    new, unmapped = map_labels(users[col], rules)
    if len(unmapped):
        print(f"[{col}] NOT MAPPED - decide by hand:\n{unmapped}\n")
    new = new.fillna(old.astype("string").str.strip())        # keep unmapped as they are
    changed = int((old.notna() & (old != new)).sum())
    users[col] = new
    log_rule("2. Labels", "users", f"Standardise {col}", "Fix", before, len(users), changed,
             f"{users[col].nunique()} distinct labels after cleaning")

print("AFTER:", {c: sorted(users[c].dropna().unique().tolist()) for c in ["channel", "country", "platform", "plan"]})

## Step 3 - Remove duplicates

**Why.** Duplicated users inflate signups and every conversion rate. We define "duplicate" before deleting.

**Comment.** (a) Exact duplicate rows are removed. (b) If the same `user_id` still appears with different values, we keep the most complete row (most non-empty cells) and log it.

In [ ]:
# (a) exact duplicates
before = len(users)
mask = users.duplicated(keep="first")
users = users[~mask]
log_rule("3. Duplicates", "users", "Exact duplicate rows", "Remove", before, len(users), mask.sum(),
         "Identical in every column: double-logged records")

# (b) same user_id, different values -> keep the most complete row
before = len(users)
users = users.assign(_filled=users.notna().sum(axis=1))
users = users.sort_values(["user_id", "_filled"], ascending=[True, False])
mask = users.duplicated(subset="user_id", keep="first")
users = users[~mask].drop(columns="_filled").sort_index()
log_rule("3. Duplicates", "users", "Same user_id, different values", "Remove", before, len(users), mask.sum(),
         "Kept the most complete row for each user_id")

## Step 4 - Remove test / internal accounts

**Why.** Internal and test accounts are not customers: they distort signups, activation and conversion.

**Comment.** The file has no email column, so we look at `user_id` patterns and at abnormal behaviour. **Inspect the matches before deleting** and adapt the pattern to what the data really shows.

In [ ]:
# 1) look first: which user_ids look non-standard?
pattern = r"(?i)test|internal|demo|qa|admin|fitloop|staff"
suspects = users[users["user_id"].str.contains(pattern, na=False)]
print(len(suspects), "suspect accounts")
suspects.head(10)

In [ ]:
before = len(users)
mask = users["user_id"].str.contains(pattern, na=False)
users = users[~mask]
log_rule("4. Test accounts", "users", f"user_id matches /{pattern}/", "Remove", before, len(users), mask.sum(),
         "Test / internal accounts: not real customers")

## Step 5 - Remove impossible values

**Why.** Events must happen in a logical order and inside the observation window. Rows that break this give negative lifetimes or phantom users.

**Comment.** Rules: observed window is 1 Jan 2025 - 30 Jun 2026 (anything after is not observed yet); trial cannot start before signup; payment cannot start before the trial; end cannot be before start. Each rule is counted separately.

In [ ]:
def remove_rule(df, name, mask, comment):
    mask = mask.fillna(False)
    out = df[~mask]
    log_rule("5. Impossible", "users", name, "Remove", len(df), len(out), mask.sum(), comment)
    return out

u = users
u = remove_rule(u, "signup outside Jan 2025 - Jun 2026",
                (u.signup_date < START_DATE) | (u.signup_date > EXTRACTION_DATE),
                "Outside the observation window")
u = remove_rule(u, "any date after extraction (30 Jun 2026)",
                (u.trial_start_date > EXTRACTION_DATE) | (u.subscription_start_date > EXTRACTION_DATE)
                | (u.subscription_end_date > EXTRACTION_DATE),
                "Future dates: not observed yet")
u = remove_rule(u, "trial_start before signup", u.trial_start_date < u.signup_date,
                "Trial cannot start before the account exists")
u = remove_rule(u, "paid start before trial start", u.subscription_start_date < u.trial_start_date,
                "Paying starts at the END of the trial")
u = remove_rule(u, "subscription_end before subscription_start",
                u.subscription_end_date < u.subscription_start_date,
                "Would give a negative subscription length")
users = u

# flag, not remove: paid start not ~7 days after trial start (check before deciding)
gap = (users.subscription_start_date - users.trial_start_date).dt.days
print("trial -> paid delay (days):\n", gap.describe())
print("paid users with NO trial:", int((users.subscription_start_date.notna() & users.trial_start_date.isna()).sum()))

## Step 6 - Handle missing and inconsistent values

**Why.** Dropping rows with a missing field would bias CAC and cohorts. Empty values are labelled or flagged so the user stays in the totals.

**Comment.** An **empty** `subscription_end_date` is NOT missing data: it means "still subscribed at extraction". Same for empty trial / subscription dates (= never started). We only fix columns where an empty value is truly missing.

In [ ]:
before = len(users)

# categorical fields -> 'Unknown'
for col in ["channel", "country", "platform"]:
    n = int(users[col].isna().sum())
    users[col] = users[col].fillna("Unknown")
    log_rule("6. Missing", "users", f"Missing {col} -> 'Unknown'", "Flag", before, len(users), n,
             "Kept in totals; share of Unknown reported in the diagnosis")

# activated_d7 must be 0 or 1
bad = ~users["activated_d7"].isin([0, 1])
n = int(bad.sum())
users.loc[bad, "activated_d7"] = np.nan
log_rule("6. Missing", "users", "activated_d7 not in {0,1} -> empty", "Fix", before, len(users), n,
         "Invalid flag value; excluded from activation rate")

# plan filled but user never paid -> plan should be empty
mask = users["plan"].notna() & users["subscription_start_date"].isna()
users.loc[mask, "plan"] = pd.NA
log_rule("6. Missing", "users", "plan filled but never paid -> empty", "Fix", before, len(users), mask.sum(),
         "Plan is only meaningful for paying users")

# paid but no plan -> flag
mask = users["subscription_start_date"].notna() & users["plan"].isna()
users["flag_paid_no_plan"] = mask
log_rule("6. Missing", "users", "paid user without plan", "Flag", before, len(users), mask.sum(),
         "Kept; flag_paid_no_plan = True (revenue cannot be computed exactly)")

## Step 7 - Clean the marketing spend file

**Why.** CAC = spend / paying customers, so spend must be complete and correct for every paid channel and month.

**Comment.** 4 paid channels x 18 months should give 72 rows but the file has 71, so at least one month/channel is missing. We list the gaps, we do **not** invent values: the choice (estimate or exclude) is justified in the report.

In [ ]:
before = len(spend)
spend["channel"], un = map_labels(spend["channel"], CHANNEL_RULES)
spend["month"] = parse_dates(spend["month"])
spend["month"] = spend["month"].dt.to_period("M").dt.to_timestamp()
spend["spend_eur"] = pd.to_numeric(spend["spend_eur"].str.replace(r"[^\d.\-]", "", regex=True), errors="coerce")
log_rule("7. Spend", "spend", "Standardise channel, month and spend_eur types", "Fix", before, len(spend), len(spend))

# duplicates (same month + channel)
mask = spend.duplicated(subset=["month", "channel"], keep="first")
spend = spend[~mask]
log_rule("7. Spend", "spend", "Duplicate month + channel", "Remove", before, len(spend), mask.sum())

# impossible spend
before = len(spend)
mask = spend["spend_eur"].isna() | (spend["spend_eur"] <= 0)
log_rule("7. Spend", "spend", "Missing or <= 0 spend", "Flag", before, len(spend), mask.sum(),
         "Kept for inspection: decide in the report whether to estimate or exclude")
spend["flag_bad_spend"] = mask

# gaps: expected month x channel grid
months = pd.date_range("2025-01-01", "2026-06-01", freq="MS")
expected = pd.MultiIndex.from_product([months, sorted(spend["channel"].dropna().unique())], names=["month", "channel"])
missing = expected.difference(pd.MultiIndex.from_frame(spend[["month", "channel"]]))
print("Expected rows:", len(expected), "| present:", len(spend), "| missing combos:", len(missing))
print(list(missing))
log_rule("7. Spend", "spend", "Gaps in month x channel grid", "Flag", len(spend), len(spend), len(missing),
         "Missing combinations listed above; not filled with invented data")

## Step 8 - Flag censored cohorts

**Why.** Users who signed up recently have had little time to churn or convert. Comparing them with older cohorts would make recent cohorts look better only because they are younger.

**Comment.** We compute each user's age in months at extraction. Cohorts must only be compared over the same age (e.g. month 1-3). No rows are removed.

In [ ]:
users["signup_month"] = users["signup_date"].dt.to_period("M").dt.to_timestamp()
users["age_months"] = ((EXTRACTION_DATE.year - users["signup_date"].dt.year) * 12
                       + (EXTRACTION_DATE.month - users["signup_date"].dt.month))
users["censored_3m"] = users["age_months"] < 3          # less than 3 months observed

n = int(users["censored_3m"].sum())
log_rule("8. Censoring", "users", "Flag cohorts with < 3 months of observation", "Flag",
         len(users), len(users), n, "Compare cohorts only on the same age; no rows removed")
users[["signup_month", "age_months", "censored_3m"]].head()

## Step 9 - Check and save

**Why.** A last sanity check on the sizes, then we save the clean files used for the diagnosis and the growth model.

In [ ]:
print(f"users: raw {len(users_raw):,} -> clean {len(users):,} rows")
print(f"spend: raw {len(spend_raw):,} -> clean {len(spend):,} rows")

users.to_csv("users_clean.csv", index=False)
spend.to_csv("marketing_spend_clean.csv", index=False)

### Summary of the cleaning
*(Fill in once the notebook has run.)*

- **Raw:** __ user rows and __ spend rows. **Clean:** __ user rows and __ spend rows.
- **Main problems found:** duplicates, test accounts, inconsistent labels, mixed date formats, impossible dates, missing values, gaps in the spend file.
- **Key choices:** what we removed vs only flagged, how we treated spend gaps, and why an empty `subscription_end_date` means "still active".